# Logistic Regression — Will this person have diabetes? (yes / no)

**What is this?**  
We want the computer to guess **yes or no**: does this person have diabetes?

**Simple idea:**  
Linear Regression draws a straight line and guesses a **number**.  
Logistic Regression still uses a line, then **squeezes** the answer into a probability between 0 and 1 (the **S-shaped sigmoid**).

- Probability ≥ 0.5 → we say **YES (1)**  
- Probability < 0.5 → we say **NO (0)**

**Type:** Supervised → **Classification** (labels, not prices).

Dataset: `diabetes.csv` (Pima Indians Diabetes) already in your folder.

Typical columns: Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age, **Outcome** (0 = no, 1 = yes).

## STEP 1 — Import libraries

We borrow ready-made tools.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay
)

sns.set_style('whitegrid')
print('Libraries loaded. Ready!')

# Warnings
import warnings
warnings.filterwarnings('ignore')

Libraries loaded. Ready!


## STEP 2 — Load the dataset

In [ ]:
df = pd.read_csv('diabetes.csv')

print('Rows, columns:', df.shape)
df.head()

## STEP 3 — EDA (look first)

We check missing values, how many yes/no, and which features relate to Outcome.

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# How many 0 vs 1? (class balance)
print(df['Outcome'].value_counts())
print('\nPercent:')
print(df['Outcome'].value_counts(normalize=True).round(3))

In [ ]:
# In this famous dataset, 0 often means "not measured" for some columns
maybe_missing = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print('How many zeros (often = missing):')
for c in maybe_missing:
    if c in df.columns:
        print(c, (df[c] == 0).sum())

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Which features move with Outcome?')
plt.show()

In [ ]:
# Glucose is usually the strongest simple picture
if 'Glucose' in df.columns:
    plt.figure(figsize=(7, 4))
    sns.boxplot(x='Outcome', y='Glucose', data=df)
    plt.title('Higher glucose → more diabetes?')
    plt.show()

## STEP 4 — Preprocessing

1. Treat impossible zeros as missing, fill with **median**.  
2. **X** = features, **y** = Outcome (0/1).

In [ ]:
target = 'Outcome'
if target not in df.columns:
    print('Columns:', list(df.columns))
    raise ValueError('Set target to your 0/1 column name.')

zero_as_missing = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for c in zero_as_missing:
    if c in df.columns:
        df[c] = df[c].replace(0, np.nan)
        df[c] = df[c].fillna(df[c].median())
        print('Cleaned zeros in', c)

X = df.drop(columns=[target])
y = df[target]

print('Features:', list(X.columns))
print('We predict:', target, '(0 = no diabetes, 1 = diabetes)')

## STEP 5 — Train / test split

Study on 80%, test on 20% we never showed the model.

**YES, use stratify=y** so train and test have a similar mix of yes/no.  
(We did **not** do this for Linear Regression.)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train:', len(X_train), ' Test:', len(X_test))
print('Train yes-rate:', y_train.mean().round(3))
print('Test  yes-rate:', y_test.mean().round(3))

## STEP 6 — Feature scaling (recommended)

Glucose, BMI, Age have different sizes. Scaling puts them on a similar scale so the model is fair and trains better.

- `fit_transform` on **train** only  
- `transform` on **test** (same ruler, no cheating)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print('Scaling done.')

## STEP 7 — Train the model

The model learns **weights**. Big positive weight → that feature pushes toward **YES**.  
`C` is like “how strictly we keep weights small” (regularization). Default is fine for class.

In [ ]:
model = LogisticRegression(max_iter=1000, C=1.0)
model.fit(X_train_scaled, y_train)

print('Starting value (intercept):', model.intercept_[0])
print('\nWeight for each feature ( + means more chance of diabetes ):')
for name, w in zip(X.columns, model.coef_[0]):
    print(f'  {name:28s} {w:8.3f}')

## STEP 8 — Predict

- `predict` → 0 or 1  
- `predict_proba` → chance of diabetes (0.0 to 1.0)

In [ ]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]  # chance of class 1

peek = pd.DataFrame({
    'Real': y_test.values[:10],
    'Guess': y_pred[:10],
    'P(diabetes)': y_prob[:10].round(3)
})
peek

## STEP 9 — Evaluation (classification scores)

| Score | Plain English |
|--------|----------------|
| Accuracy | How many guesses were right? |
| Precision | When we say YES, how often are we right? |
| Recall | Of all real YES people, how many did we catch? |
| F1 | Balance of precision and recall |
| ROC-AUC | How well we rank sick vs healthy (1.0 = perfect) |

Confusion matrix:  
TN / FP  
FN / TP

In [ ]:
print('Accuracy :', round(accuracy_score(y_test, y_pred), 3))
print('Precision:', round(precision_score(y_test, y_pred), 3))
print('Recall   :', round(recall_score(y_test, y_pred), 3))
print('F1       :', round(f1_score(y_test, y_pred), 3))
print('ROC-AUC  :', round(roc_auc_score(y_test, y_prob), 3))
print()
print(classification_report(y_test, y_pred, target_names=['No diabetes', 'Diabetes']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Guess NO', 'Guess YES'],
            yticklabels=['Real NO', 'Real YES'])
plt.title('Confusion matrix')
plt.show()

In [ ]:
RocCurveDisplay.from_estimator(model, X_test_scaled, y_test)
plt.title('ROC curve (higher curve = better ranking)')
plt.show()

## STEP 10 — Optional: tune C

`C` is the main knob. Smaller C = stronger “keep weights small” (less overfitting).

In [ ]:
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(
    LogisticRegression(max_iter=1000),
    {'C': [0.01, 0.1, 1, 10, 100]},
    cv=5,
    scoring='roc_auc'
)
search.fit(X_train_scaled, y_train)
print('Best C:', search.best_params_)
print('Best CV ROC-AUC:', round(search.best_score_, 3))
print('Test ROC-AUC:', round(roc_auc_score(y_test, search.predict_proba(X_test_scaled)[:, 1]), 3))

## Key notes (remember these)

| Topic | Logistic Regression |
|--------|---------------------|
| Task | Yes/no (or few classes) |
| Scaling | **Recommended** |
| Stratify | **YES** |
| Output | Probability → then 0/1 at threshold 0.5 |
| Metrics | Accuracy, Precision, Recall, F1, ROC-AUC, confusion matrix |
| Tune | mainly `C` |
| vs Linear | Linear guesses a **number**; Logistic guesses a **chance** then a **label** |

**One sentence for the class:**  
We turn medical numbers into a probability of diabetes, then say YES if that chance is high enough, and we check how many sick and healthy people we got right.